In [ ]:
# CELL 0 - Environment Check
# ─────────────────────────────────────────────────────────────────────────────
# PrecisionSkin - ECZEMA QUALITY CHECK (same layout as the facial-only notebook).
# Question: how many eczema lesions are STILL MISSING from the boxes we already corrected by hand (the earlier eczema review)?
#
#   CELL 2  picks N random corrected photos          CELL 3  makes the package for the checker (a groupmate)
#   (she opens them in the box tool WITH the existing boxes, adds the lesions that have no box, removes wrong ones, presses S on EVERY photo)
#   CELL 4  progress                                  CELL 5  RESULT: how many boxes were added / removed / kept
#
# Your original photos and labels are never changed (copies only). Nothing is uploaded anywhere.
# Kernel: any WSL kernel with pandas / numpy (the .venv works).
# ─────────────────────────────────────────────────────────────────────────────
import os, sys, json, time, shutil, random, zipfile, csv
from pathlib import Path
import numpy as np, pandas as pd
print("Python:", sys.version.split()[0], "| kernel:", sys.executable)
print("✓ imports OK")

In [ ]:
# CELL 1 - Configuration
# ─────────────────────────────────────────────────────────────────────────────
#   CHECKER   : who does the check (no spaces; used in the folder name)
#   N_QA      : how many photos she re-checks (60 is enough to see whether many lesions are missing)
#   REVIEW_ALL: the folder of the earlier eczema review (images + your corrected labels)
#   QA_ROOT   : where the package and the saved copies are made (Windows side)
# ─────────────────────────────────────────────────────────────────────────────
CHECKER = "Francesca"
N_QA = 60
SEED = 42

REPO = Path("/mnt/c/Users/Bautista/source/repos/SkinDevApp")
REVIEW_ALL = Path("/mnt/c/Users/Bautista/Documents/PrecisionSkin_EczemaReview/ALL")
QA_ROOT = Path("/mnt/c/Users/Bautista/Documents/PrecisionSkin_EczemaQA")
QA_ROOT.mkdir(parents=True, exist_ok=True)
random.seed(SEED); np.random.seed(SEED)

def to_win(p):
    s = str(p)
    return ("C:\\" + s[len("/mnt/c/"):].replace("/", "\\")) if s.startswith("/mnt/c/") else s

def read_boxes(p):
    return [tuple(float(v) for v in l.split()[1:5]) for l in Path(p).read_text().splitlines() if len(l.split()) >= 5] if Path(p).exists() else []

print("checker:", CHECKER, "| photos to re-check:", N_QA)
print("✓ configuration set")

In [ ]:
# CELL 2 - Pick the Photos to Re-check
# ─────────────────────────────────────────────────────────────────────────────
# From the earlier eczema review: photos that were reviewed (saved after the machine drafts were made) and that have at least one box.
# A random sample, always the same one for the same SEED.
# ─────────────────────────────────────────────────────────────────────────────
meta_old = json.load(open(REVIEW_ALL.parent / "_drafts" / "meta.json"))["drafted_at"]
IMG_EXT = {".jpg", ".jpeg", ".png", ".webp", ".bmp"}
rows = []
for img in sorted((REVIEW_ALL / "images").iterdir()):
    if img.suffix.lower() not in IMG_EXT:
        continue
    lab = REVIEW_ALL / "labels" / (img.stem + ".txt")
    if lab.exists() and lab.stat().st_mtime > meta_old + 1 and read_boxes(lab):
        rows.append(dict(name=img.name, path=str(img), n_boxes=len(read_boxes(lab))))
pool = pd.DataFrame(rows)
print("reviewed photos with at least one box:", len(pool), "| boxes in them:", int(pool.n_boxes.sum()), "| mean per photo: %.1f" % pool.n_boxes.mean())
qa = pool.sample(min(N_QA, len(pool)), random_state=SEED).reset_index(drop=True)
print("\nphotos picked for the quality check:", len(qa), "| your boxes in them:", int(qa.n_boxes.sum()))
qa.to_csv(QA_ROOT / "qa_photos.csv", index=False)

In [ ]:
# CELL 3 - Make the Package for the Checker
# ─────────────────────────────────────────────────────────────────────────────
# Folder <Checker>__eczema_qa with the photos and YOUR existing boxes already drawn. A hidden copy of your boxes is kept in QA_ROOT/_original
# so the result can be measured. The zip contains the photos, the box tool and START_HERE.txt (exact commands and the job description).
# ─────────────────────────────────────────────────────────────────────────────
d = QA_ROOT / ("%s__eczema_qa" % CHECKER)
orig = QA_ROOT / "_original"
for p in (d, orig):
    if p.exists() and p is d:
        shutil.rmtree(p)
(d / "images").mkdir(parents=True); (d / "labels").mkdir(parents=True); orig.mkdir(exist_ok=True)
(d / "classes.txt").write_text("eczema\n")
for r in qa.itertuples():
    shutil.copy2(r.path, d / "images" / r.name)
    lab = (REVIEW_ALL / "labels" / (Path(r.name).stem + ".txt")).read_text()
    (d / "labels" / (Path(r.name).stem + ".txt")).write_text(lab)          # your boxes, written now (so they count as 'not yet checked')
    (orig / (Path(r.name).stem + ".txt")).write_text(lab)
time.sleep(1.2)
meta = QA_ROOT / "_drafts" / "meta.json"
(QA_ROOT / "_drafts").mkdir(exist_ok=True)
json.dump({"drafted_at": time.time(), "note": "quality-check folders created"}, open(meta, "w"))

txt = ["ECZEMA QUALITY CHECK (about 30-45 minutes)", "",
       "These photos already have boxes (drawn by Elaiza). Your job: find the eczema lesions that have NO box and add them.", "",
       "1. You only need Python 3.8 or newer (python.org, tick 'Add python to PATH'). Nothing else to install.",
       "2. Unzip this package, open PowerShell in the unzipped folder and run:", "",
       '   py -3 lesion_box_tool.py "%s__eczema_qa" --label eczema --annotator %s --port 8800' % (CHECKER, CHECKER), "",
       "3. For EVERY photo:",
       "   - Look at the whole photo. Press H to hide the boxes if you want to see the skin; E = contrast view; mouse wheel = zoom.",
       "   - Add a box (drag) around every patch of eczema (red, scaly, dry or inflamed skin) that has no box.",
       "   - Delete a box (click it, then Delete) only if it is clearly NOT eczema. Fix a box only if it is clearly wrong.",
       "   - Do NOT redraw boxes that are fine. Do not change your mind about borderline cases that are already boxed.",
       "   - Press S to save and go to the next photo (even when you changed nothing). X = no eczema visible at all.",
       "4. You can stop and continue later: saved photos stay saved. Press Ctrl+C in PowerShell to stop the tool.",
       "5. When finished, zip the folder '%s__eczema_qa' and send it back. Do not rename any file." % CHECKER]
zp = QA_ROOT / ("send_to_%s_eczema_qa.zip" % CHECKER)
with zipfile.ZipFile(zp, "w", zipfile.ZIP_DEFLATED) as z:
    z.writestr("START_HERE.txt", "\n".join(txt)); z.write(REPO / "lesion_box_tool.py", "lesion_box_tool.py"); z.write(meta, "_drafts/meta.json")
    for f in d.rglob("*"):
        if f.is_file():
            z.write(f, str(f.relative_to(QA_ROOT)))
print("package ready: %s (%.1f MB, %d photos)" % (zp, zp.stat().st_size / 1e6, len(qa)))
print("\nYour own command to try it (Windows PowerShell inside the SkinDevApp folder):")
print('  py -3.13 lesion_box_tool.py "%s" --label eczema --annotator %s --port 8800' % (to_win(d), CHECKER))
print("\nWhen she sends the folder back, put it into", QA_ROOT, "(replace the matching folder) and run CELL 4 and CELL 5.")

In [ ]:
# CELL 4 - Progress
# ─────────────────────────────────────────────────────────────────────────────
# How many photos the checker has saved so far (a photo counts only after she presses S on it).
# ─────────────────────────────────────────────────────────────────────────────
d = QA_ROOT / ("%s__eczema_qa" % CHECKER)
t_ref = json.load(open(QA_ROOT / "_drafts" / "meta.json"))["drafted_at"]
saved = [r.name for r in qa.itertuples() if (d / "labels" / (Path(r.name).stem + ".txt")).exists() and (d / "labels" / (Path(r.name).stem + ".txt")).stat().st_mtime > t_ref + 1]
print("saved: %d / %d photos" % (len(saved), len(qa)))
lg = d / "labeling_log.csv"
if lg.exists():
    last = {}
    for r in csv.DictReader(open(lg, encoding="utf-8")):
        try:
            if r.get("action") in ("save", "none"): last[r["image"]] = float(r["seconds"])
        except Exception: pass
    if last:
        print("average %.0f seconds per photo" % np.mean(list(last.values())))

In [ ]:
# CELL 5 - RESULT: How Complete Were the Corrected Boxes?
# ─────────────────────────────────────────────────────────────────────────────
# Compares your original boxes with the boxes after the check (a box matches when the overlap, IoU, is at least 0.5):
#   kept    = your box is still there      removed = she deleted it      added = a lesion she found that had NO box
# MISSING RATE = added / (kept + added): the share of the final boxes that were missing before. This is the number to look at.
# ─────────────────────────────────────────────────────────────────────────────
def iou(a, b):
    ax0, ay0, ax1, ay1 = a[0] - a[2] / 2, a[1] - a[3] / 2, a[0] + a[2] / 2, a[1] + a[3] / 2
    bx0, by0, bx1, by1 = b[0] - b[2] / 2, b[1] - b[3] / 2, b[0] + b[2] / 2, b[1] + b[3] / 2
    iw, ih = max(0, min(ax1, bx1) - max(ax0, bx0)), max(0, min(ay1, by1) - max(ay0, by0)); i = iw * ih
    u = a[2] * a[3] + b[2] * b[3] - i
    return i / u if u > 0 else 0.0

def compare(orig_boxes, final_boxes):
    used, kept = set(), 0
    for o in orig_boxes:
        best, bj = 0.0, None
        for j, f in enumerate(final_boxes):
            if j in used: continue
            v = iou(o, f)
            if v > best: best, bj = v, j
        if bj is not None and best >= 0.5:
            used.add(bj); kept += 1
    return kept, len(orig_boxes) - kept, len(final_boxes) - kept

rows = []
for r in qa.itertuples():
    p = d / "labels" / (Path(r.name).stem + ".txt")
    if not (p.exists() and p.stat().st_mtime > t_ref + 1):
        continue
    k, rem, add = compare(read_boxes(QA_ROOT / "_original" / (Path(r.name).stem + ".txt")), read_boxes(p))
    rows.append(dict(photo=r.name, original=k + rem, kept=k, removed=rem, added=add))
res = pd.DataFrame(rows)
if res.empty:
    print("No saved photos yet: wait for the checker's folder, then re-run CELL 4 and CELL 5.")
else:
    res.to_csv(QA_ROOT / "qa_result_per_photo.csv", index=False)
    kept, rem, add = int(res.kept.sum()), int(res.removed.sum()), int(res.added.sum())
    final = kept + add
    print("photos checked: %d of %d" % (len(res), len(qa)))
    print("your boxes in them: %d | kept: %d | removed (not eczema / wrong): %d | ADDED (were missing): %d" % (res.original.sum(), kept, rem, add))
    print("photos where she added at least one box: %d of %d (%.0f %%)" % ((res.added > 0).sum(), len(res), 100 * (res.added > 0).mean()))
    print("added boxes per checked photo: mean %.1f, max %d" % (res.added.mean(), res.added.max()))
    miss = add / max(1, final); wrong = rem / max(1, kept + rem)
    print("\nMISSING RATE (share of the final boxes that were missing before): %.0f %%" % (100 * miss))
    print("WRONG-BOX RATE (share of your boxes she removed): %.0f %%" % (100 * wrong))
    if miss > 0.20:
        print("\nVERDICT: MANY lesions are missing. Re-check more of the %d corrected photos (about %d of them would need work)." % (len(pool), len(pool) * (res.added > 0).mean()))
    elif miss > 0.10:
        print("\nVERDICT: a fair number of lesions are missing. Consider re-checking the photos where lesions are small or numerous.")
    else:
        print("\nVERDICT: the corrected boxes are mostly complete (missing rate under 10 %). No big re-check needed.")